# Figures

This notebook **only calls into `src/report/`**. No analysis logic lives here.

Why: a notebook-only code path cannot be run by `scripts/run_pipeline.py`, cannot be
tested, and cannot be reproduced on Euler. If you find yourself writing a metric or a
plotting detail in a cell, it belongs in `src/report/figures.py` where it has a test and
a CLI.

Use this to look at figures interactively while tuning them. The real figures are produced by:

```bash
python -m src.report.figures --in results/run1/leaderboard.csv --out results/run1/figures \
    --target configs/target.yaml --plddt-dir results/run1/fold --structure-dir results/run1/fold
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
%matplotlib inline

from src.report import figures as fig
from src.report import fake_data

# Point this at a real run; the fake table is the fallback so the notebook
# always renders, and anything drawn from it is stamped FAKE DATA.
LEADERBOARD = Path('../results/run1/leaderboard.csv')
PLDDT_DIR = Path('../results/run1/fold')
STRUCTURE_DIR = Path('../results/run1/fold')
TARGET = Path('../configs/target.yaml')

if LEADERBOARD.exists():
    df = pd.read_csv(LEADERBOARD)
    print(f'{len(df)} designs from {LEADERBOARD}')
else:
    df = fake_data.make_fake_leaderboard()
    print(f'{LEADERBOARD} not found - using FAKE data ({len(df)} rows)')
df.head(3)

## Figure 1 — designs vs. the wild-type baseline

In [ ]:
fig.fig_score_distributions(df)

## Figure 2 — novelty vs. predicted quality

The top-right quadrant is the pitch: novel *and* confident.

In [ ]:
fig.fig_novelty_vs_quality(df)

## Figure 3 — the best design

Needs a folded structure and a per-residue pLDDT file; panels say so when they are missing.

In [ ]:
fig.fig_best_design(
    df,
    plddt_dir=PLDDT_DIR if PLDDT_DIR.exists() else None,
    target_config=TARGET if TARGET.exists() else None,
    structure_dir=STRUCTURE_DIR if STRUCTURE_DIR.exists() else None,
)

## Save all three

Equivalent to the CLI above; writes PNG (300 dpi) and PDF.

In [ ]:
if LEADERBOARD.exists():
    for p in fig.build_all(LEADERBOARD, '../results/run1/figures',
                           plddt_dir=PLDDT_DIR, target_config=TARGET,
                           structure_dir=STRUCTURE_DIR):
        print(p)
else:
    print('no real leaderboard yet - not writing figures from fake data')